In [3]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 5.6 MB/s  0:00:02m 4.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 9.8 MB/s  0:00:0011.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
df = pd.read_csv("2017-1-1 to 2026-5-1 weather data.csv", encoding="utf-8")
print("Basic Dataset Information: \n")
print(f"Data shape (rows, columns): {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())

Basic Dataset Information: 

Data shape (rows, columns): (3408, 11)

First 5 rows:
                  date  tavg  tmin  tmax  prcp  snow  wdir  wspd  wpgt  \
0  2017-01-01 00:00:00  -4.4  -6.0   0.0   0.0   NaN   NaN   4.8   NaN   
1  2017-01-02 00:00:00  -1.2  -8.0   8.0   0.0   NaN   NaN   7.5   NaN   
2  2017-01-03 00:00:00  -1.7  -8.0   9.0   NaN   NaN   NaN   5.1   NaN   
3  2017-01-04 00:00:00  -0.1  -9.0   5.0   0.0   NaN   NaN   5.6   NaN   
4  2017-01-05 00:00:00  -0.9  -3.0   3.6   NaN   NaN   NaN   6.9   NaN   

     pres  tsun  
0  1025.4   NaN  
1  1025.6   NaN  
2  1022.8   NaN  
3  1026.3   NaN  
4  1030.6   NaN  

Column names:
['date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']


In [7]:
# Convert date column to standard datetime format
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# Define seasons
def get_season(month):
    if month in [2,3,4]:
        return "Spring"
    elif month in [5,6,7]:
        return "Summer"
    elif month in [8,9,10]:
        return "Autumn"
    else:
        return "Winter"

df['season'] = df['month'].apply(get_season)

# Mark bird breeding season
def is_breeding(month):
    return 1 if 3 <= month <= 7 else 0

df['breeding_season'] = df['month'].apply(is_breeding)

# Check new time features
print("==== Time Features Created ===\n")
print(df[['date', 'year', 'month', 'season', 'breeding_season']].head(10))

==== Time Features Created ===

        date  year  month  season  breeding_season
0 2017-01-01  2017      1  Winter                0
1 2017-01-02  2017      1  Winter                0
2 2017-01-03  2017      1  Winter                0
3 2017-01-04  2017      1  Winter                0
4 2017-01-05  2017      1  Winter                0
5 2017-01-06  2017      1  Winter                0
6 2017-01-07  2017      1  Winter                0
7 2017-01-08  2017      1  Winter                0
8 2017-01-09  2017      1  Winter                0
9 2017-01-10  2017      1  Winter                0


In [9]:
# Convert date column to standard datetime format
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# Define seasons
def get_season(month):
    if month in [2,3,4]:
        return "Spring"
    elif month in [5,6,7]:
        return "Summer"
    elif month in [8,9,10]:
        return "Autumn"
    else:
        return "Winter"

df['season'] = df['month'].apply(get_season)

# Mark bird breeding season
def is_breeding(month):
    return 1 if 3 <= month <= 7 else 0

df['breeding_season'] = df['month'].apply(is_breeding)

# Check new time features
print("==== Time Features Created ===\n")
print(df[['date', 'year', 'month', 'season', 'breeding_season']].head(10))

==== Time Features Created ===

        date  year  month  season  breeding_season
0 2017-01-01  2017      1  Winter                0
1 2017-01-02  2017      1  Winter                0
2 2017-01-03  2017      1  Winter                0
3 2017-01-04  2017      1  Winter                0
4 2017-01-05  2017      1  Winter                0
5 2017-01-06  2017      1  Winter                0
6 2017-01-07  2017      1  Winter                0
7 2017-01-08  2017      1  Winter                0
8 2017-01-09  2017      1  Winter                0
9 2017-01-10  2017      1  Winter                0


In [11]:
# CMissing values and percentage
print("Missing Value Statistics: \n")
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": round(missing_pct, 2)
})
print(missing_summary)

# 1.Precipitation & snow: missing = no precipitation, fill with 0
df['prcp'] = df['prcp'].fillna(0)
df['snow'] = df['snow'].fillna(0)

# 2.Temperature, wind speed, pressure: linear interpolation
df['tavg'] = df['tavg'].interpolate(method='linear')
df['tmin'] = df['tmin'].interpolate(method='linear')
df['tmax'] = df['tmax'].interpolate(method='linear')
df['wspd'] = df['wspd'].interpolate(method='linear')
df['pres'] = df['pres'].interpolate(method='linear')

# 3.Drop columns with excessive missing values
drop_cols = ['wdir', 'wpgt', 'tsun']
df = df.drop(columns=drop_cols, errors='ignore')

# 4.Verify no missing values left
print(f"\nRemaining Missing Values: {df.isnull().sum().sum()}")

Missing Value Statistics: 

                 Missing Count  Missing Percentage (%)
date                         0                     0.0
tavg                         0                     0.0
tmin                         0                     0.0
tmax                         0                     0.0
prcp                         0                     0.0
snow                         0                     0.0
wspd                         0                     0.0
pres                         0                     0.0
year                         0                     0.0
month                        0                     0.0
day                          0                     0.0
season                       0                     0.0
breeding_season              0                     0.0

Remaining Missing Values: 0


In [12]:
weather_vars = ['tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd', 'pres']
# Identify ourliers by using IQR
def remove_outliers(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]

# Remove outliers
for col in weather_vars:
    if col in df.columns:
        df = remove_outliers(df, col)
print("After Outlier Removal\n")
print(f"Final data shape: {df.shape}")
print(df.describe())

After Outlier Removal

Final data shape: (2614, 13)
                             date         tavg         tmin         tmax  \
count                        2614  2614.000000  2614.000000  2614.000000   
mean   2021-08-26 14:20:28.462127    12.522341     5.914308    19.586381   
min           2017-01-01 00:00:00   -13.200000   -23.000000    -7.000000   
25%           2019-04-14 06:00:00     1.700000    -4.575000     9.000000   
50%           2021-09-17 12:00:00    12.800000     5.800000    20.400000   
75%           2023-12-17 18:00:00    23.000000    16.200000    30.000000   
max           2026-05-01 00:00:00    32.900000    28.900000    42.000000   
std                           NaN    11.453948    11.582327    11.345457   

         prcp    snow         wspd        pres         year        month  \
count  2614.0  2614.0  2614.000000  2614.00000  2614.000000  2614.000000   
mean      0.0     0.0     8.705681  1017.52280  2021.170620     6.306427   
min       0.0     0.0     1.300000 

In [13]:
# 1. Temperature range
df['temp_range'] = df['tmax'] - df['tmin']

# 2. Extreme temperature
df['extreme_high'] = (df['tmax'] >= 30).astype(int)
df['extreme_low'] = (df['tmin'] <= 0).astype(int)

# 3. Precipitation day
df['rain_day'] = (df['prcp'] > 0).astype(int)

# 4. Strong wind (adjusting threshold)
df['strong_wind'] = (df['wspd'] >= 10).astype(int)

# 5. Growing Degree Days (adjusting base tem)
base_temp = 5
df['GDD'] = df['tavg'].apply(lambda x: max(x - base_temp, 0))

print("New Ecological Features: \n")
print(df[['tavg','temp_range','GDD','extreme_high','rain_day','strong_wind']].head())

New Ecological Features: 

   tavg  temp_range  GDD  extreme_high  rain_day  strong_wind
0  -4.4         6.0  0.0             0         0            0
1  -1.2        16.0  0.0             0         0            0
2  -1.7        17.0  0.0             0         0            0
3  -0.1        14.0  0.0             0         0            0
4  -0.9         6.6  0.0             0         0            0


In [14]:
# 1. Daily cleaned data
df_daily = df.copy()

# 2. Monthly aggregation
df_monthly = df.groupby(['year', 'month']).agg({
    'tavg': 'mean',
    'tmin': 'mean',
    'tmax': 'mean',
    'prcp': 'sum',
    'snow': 'sum',
    'wspd': 'mean',
    'pres': 'mean',
    'GDD': 'sum',
    'extreme_high': 'sum',
    'extreme_low': 'sum',
    'rain_day': 'sum',
    'strong_wind': 'sum'
}).reset_index()

# 3. Seasonal aggregation
df_seasonal = df.groupby(['year', 'season']).agg({
    'tavg': 'mean',
    'prcp': 'sum',
    'GDD': 'sum',
    'extreme_high': 'sum'
}).reset_index()

# 4. Yearly aggregation
df_yearly = df.groupby('year').agg({
    'tavg': 'mean',
    'tmin': 'mean',
    'tmax': 'mean',
    'prcp': 'sum',
    'GDD': 'sum',
    'extreme_high': 'sum',
    'extreme_low': 'sum'
}).reset_index()

print("Aggregated Data: \n")
print("Monthly data:")
print(df_monthly.head())
print("\nSeasonal data:")
print(df_seasonal.head())
print("\nYearly data:")
print(df_yearly.head())

Aggregated Data: 

Monthly data:
   year  month       tavg       tmin       tmax  prcp  snow       wspd  \
0  2017      1  -2.253846  -7.842308   3.350000   0.0   0.0   8.607692   
1  2017      2   2.466667  -4.225000   8.858333   0.0   0.0  10.175000   
2  2017      3   9.204348   1.895652  15.439130   0.0   0.0  10.273913   
3  2017      4  16.946429   8.757143  23.685714   0.0   0.0  10.421429   
4  2017      5  23.791667  15.345833  30.525000   0.0   0.0  10.712500   

          pres    GDD  extreme_high  extreme_low  rain_day  strong_wind  
0  1029.488462    0.0             0           26         0            8  
1  1025.491667   10.7             0           22         0           12  
2  1021.102174   99.9             0            8         0           10  
3  1012.335714  334.5             3            0         0           12  
4  1009.125000  451.0            13            0         0           15  

Seasonal data:
   year  season       tavg  prcp     GDD  extreme_high
0  2017

In [15]:
# 保存路径：电脑桌面下的 Weather_Processed 文件夹
# 自动识别系统桌面路径
from pathlib import Path
desktop_path = str(Path.home() / "Desktop")
save_path = os.path.join(desktop_path, "Weather_Processed/")

# 创建文件夹，已存在不会报错
os.makedirs(save_path, exist_ok=True)

# 导出清洗后的表格到桌面文件夹
df_daily.to_csv(os.path.join(save_path, "daily_weather_cleaned.csv"), index=False)
df_monthly.to_csv(os.path.join(save_path, "monthly_weather.csv"), index=False)
df_seasonal.to_csv(os.path.join(save_path, "seasonal_weather.csv"), index=False)
df_yearly.to_csv(os.path.join(save_path, "yearly_weather.csv"), index=False)

# 打印保存位置
print(f"All processed files saved to desktop folder: {save_path}")

NameError: name 'os' is not defined